# Scenario Playground Step 4: Sampling Design

This offline experiment compares two deterministic row indexes built from the accepted Step 3 caches:

1. one row at the final observed market session of each week; and
2. one row for each unique session when a FOMC, CPI, unemployment, GDP, or manufacturing event becomes usable.

Success means both candidates support the fixed 21-trading-session horizon, preserve pre-shock state timing, retain every structured event, and provide enough rows for later Ridge and shallow XGBoost experiments. This notebook defines timing only; it does not calculate shocks, state features, targets, splits, or models.

In [ ]:
# Setup: find the project root and build from local caches only.
from pathlib import Path
import json
import sys

import pandas as pd

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in (working_dir, *working_dir.parents)
    if (path / "scenario_modeling" / "src" / "sampling.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.sampling import build_sampling_artifacts

artifacts = build_sampling_artifacts()
summary = artifacts["summary"]
print("Construction mode:", summary["construction_mode"])
print("Shared period:", summary["shared_period"])
print("Selected design:", summary["selected_design"])


## Calendar and row timing

The canonical calendar is the set of dates with a non-missing `Close` in the selected Yahoo `^GSPC` cache. It is preferable to a calendar-day approximation because it records the sessions that actually occurred, including special closures and holiday exceptions.

The conceptual time `t` remains the completed close immediately before the modeled shock. `sample_date` is the date key when all shocks attached to a row are usable. State and target baseline both use the pre-shock close; the target endpoint is exactly the 21st later canonical session. Therefore the eventual cumulative target can include the initial imposed shock, as required by the model contract.

In [ ]:
calendar_summary = pd.Series(summary["canonical_calendar"], name="value")
timing_summary = pd.Series(summary["model_row_timing"], name="rule")
print("Canonical calendar")
print(calendar_summary.to_string())
print("\nModel-row timing")
print(timing_summary.to_string())


## Candidate size and coverage

A row is label-eligible only when the exact 21st future `^GSPC` session is within the shared selected-target coverage and every selected target field exists at both the pre-shock baseline and endpoint. Missing target values are not filled or shifted.

In [ ]:
candidate_rows = []
for key, label in (("weekly", "WEEKLY"), ("event_aware", "EVENT_AWARE")):
    candidate = summary[key]
    candidate_rows.append({
        "design": label,
        "raw_rows": candidate["raw_rows"],
        "label_eligible_rows": candidate["label_eligible_rows"],
        "21_session_boundary_removals": candidate["rows_removed_by_21_session_boundary"],
        "target_missingness_removals": candidate["rows_removed_by_endpoint_target_missingness"],
        "first_sample": candidate["first_sample"],
        "last_eligible_sample": candidate["last_label_eligible_sample"],
    })
candidate_table = pd.DataFrame(candidate_rows)
print(candidate_table.to_string(index=False))

period_rows = []
for key, label in (("weekly", "WEEKLY"), ("event_aware", "EVENT_AWARE")):
    for period, values in summary[key]["planning_period_estimates"].items():
        period_rows.append({"design": label, "period": period, **values})
print("\nPlanning-only 60/20/20 counts (not final splits)")
print(pd.DataFrame(period_rows).to_string(index=False))


## Event representation and same-day collisions

CPI and unemployment dates are the first non-missing ALFRED vintage appearances, GDP dates are the BEA advance-release dates in the Atlanta Fed TrackRecord, and FOMC/PMI dates come from their structured cached records. Observation months are retained only as identities.

If several events become usable on one session, the event-aware design keeps one row and stores every event in deterministic order. No event is discarded.

In [ ]:
representation = pd.DataFrame(summary["event_representation"]).T
print(representation.to_string())

collisions = summary["event_aware"]["same_day_collisions"]
collision_overview = {
    key: value
    for key, value in collisions.items()
    if key != "collision_details"
}
print("\nCollision overview")
print(json.dumps(collision_overview, indent=2, sort_keys=True))
print("\nFirst five collision dates")
print(pd.DataFrame(collisions["collision_details"]).head().to_string(index=False))


## Spacing and target overlap

The target windows are long relative to both row frequencies. The overlap statistic asks whether consecutive pre-shock baselines are fewer than 21 canonical sessions apart. This is not a model score; it measures why a later chronological split needs a purge/embargo.

In [ ]:
comparison_rows = []
for key, label in (("weekly", "WEEKLY"), ("event_aware", "EVENT_AWARE")):
    spacing = summary[key]["spacing"]
    overlap = summary[key]["target_overlap"]
    comparison_rows.append({
        "design": label,
        "minimum_session_gap": spacing["trading_session_gap"]["minimum"],
        "median_session_gap": spacing["trading_session_gap"]["median"],
        "maximum_session_gap": spacing["trading_session_gap"]["maximum"],
        "consecutive_session_rows": spacing["consecutive_session_rows"],
        "overlap_proportion": overlap["overlap_proportion"],
        "mean_overlapping_sessions": overlap["mean_overlapping_future_sessions"],
    })
print(pd.DataFrame(comparison_rows).to_string(index=False))
print("\nEvent-aware spacing buckets")
print(json.dumps(summary["event_aware"]["spacing"]["distribution"], indent=2))


## Point-in-time checks

These checks validate timing metadata, not feature values. In particular, a market close is not treated as state after the event, initial-vintage paths remain explicit, and every nonblank endpoint is exactly 21 observed sessions after its baseline.

In [ ]:
checks = pd.Series(summary["point_in_time_validation"], name="passed")
print(checks.to_string())
assert checks.all(), "A point-in-time validation failed."
assert not artifacts["selected"]["sample_date"].duplicated().any()
assert artifacts["selected"]["label_eligible"].all()
print("\nAll point-in-time and selected-index checks passed.")


## Decision: EVENT_AWARE

The event-aware design is selected for the MVP. It retains 585 currently label-eligible dates versus 654 weekly dates, so the sample-size cost is modest. In return, each macro shock remains attached to the session when it becomes usable, same-day events are preserved without duplicate rows, and the pre-shock state rule is much easier to explain.

The weekly design is simpler and regular, but it combines many releases inside a week, blurs their timing, and forces extra aggregation rules for a scenario tool whose inputs are explicitly shock-based. Both designs have heavily overlapping 21-session labels, so either one requires honest chronological evaluation and a horizon-length embargo later.

For a beginner: each selected row says, “Here was the market immediately before one or more known economic events; here are the event identities; later steps may calculate the shocks and ask where all 12 outputs stood exactly 21 trading sessions later.”

In [ ]:
decision = summary["final_decision"]
print("Selected:", decision["selected"])
print("Rejected:", decision["rejected"])
print("Reason:", decision["reason"])
print("\nAccepted limitations:")
for limitation in decision["accepted_limitations"]:
    print("-", limitation)

expected_outputs = [
    "data/scenario_playground/interim/sampling/normalized_event_records.csv",
    "data/scenario_playground/interim/sampling/weekly_candidate_index.csv",
    "data/scenario_playground/interim/sampling/event_aware_candidate_index.csv",
    "data/scenario_playground/interim/sampling/selected_sampling_index.csv",
    "data/scenario_playground/interim/sampling/sampling_comparison_summary.json",
]
assert all((PROJECT_ROOT / path).exists() for path in expected_outputs)
print("\nPersisted Step 4 files are present. No features, targets, splits, or models were built.")
